Run this file to Generate RotatingMNIST dataset under ./dataset/RotatingMNIST/ directory.

In [9]:
import os, sys
import torch
torch.manual_seed(0)
from tqdm.auto import trange

sys.path.append('..')
from loader import get_dataloader
from torchvision import transforms

In [10]:
data_cfg = {
    'dataset': 'MNIST',
    'root': '../dataset',
    'batch_size': 100,
    'n_workers': 4,
    'split': 'training',
    'shuffle': True,
    'digits': [3],
}

dl = get_dataloader(data_cfg)

fraction_data: None
MNIST split training | 5100
no global Laplacian, normalized K, or distance matrix is precomputed


In [11]:
data = dl.dataset.data
targets = dl.dataset.targets

In [12]:
EPISODES_PER_IMAGE = 1
TIME_HORIZON = 36
ANGLE_STEP = 10

data = data.repeat_interleave(EPISODES_PER_IMAGE, 0).unsqueeze(-1).repeat_interleave(TIME_HORIZON, -1)
targets = targets.repeat_interleave(EPISODES_PER_IMAGE, 0)

In [13]:
for i_idx in trange(data.shape[0]):
    theta_init = torch.rand(()) * 360
    for j_idx in range(data.shape[-1]):
        theta = theta_init.item() + ANGLE_STEP * j_idx
        data[i_idx, :, :, :, j_idx] = transforms.functional.affine(img=data[i_idx, :, :, :, j_idx], angle=theta, translate=[0, 0], scale=1., shear=0)
data.add_(0.1)

  0%|          | 0/5100 [00:00<?, ?it/s]

tensor([[[[[0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           ...,
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000]],

          [[0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           ...,
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000]],

          [[0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000, 0.1000,  ..., 0.1000, 0.1000, 0.1000],
           [0.1000, 0.1000

###### SAVE_PATH = '../dataset/RotatingMNIST/'
os.makedirs(SAVE_PATH, exist_ok=True)
for i in data_cfg["digits"]:
    torch.save({
        'data': data[targets == i],
        'targets': targets[targets == i]
    }, os.path.join(SAVE_PATH, f'RotatingMNIST-digit={i}.pkl'))

In [15]:
# === VERIFICATION CELL - Shows impact of upstream modifications ===

print("="*80)
print("VERIFICATION: Key Metrics (for Reactivity Testing)")
print("="*80)

print(f"\n1. Data Shape & Properties:")
print(f"   data.shape: {data.shape}")
print(f"   data.dtype: {data.dtype}")
print(f"   targets.shape: {targets.shape}")
print(f"   unique digits: {torch.unique(targets).tolist()}")

print(f"\n2. Rotation Parameters:")
print(f"   ANGLE_STEP: {ANGLE_STEP}")
print(f"   TIME_HORIZON: {TIME_HORIZON}")
print(f"   EPISODES_PER_IMAGE: {EPISODES_PER_IMAGE}")
print(f"   Expected rotations per image: {TIME_HORIZON // ANGLE_STEP} (at 360°)")

print(f"\n3. Data Statistics (before save):")
print(f"   data min: {data.min().item():.4f}")
print(f"   data max: {data.max().item():.4f}")
print(f"   data mean: {data.mean().item():.4f}")
print(f"   data std: {data.std().item():.4f}")

print(f"\n4. Output Configuration:")
print(f"   SAVE_PATH: {SAVE_PATH}")
print(f"   digits to save: {data_cfg['digits']}")
print(f"   files that will be created: {[f'RotatingMNIST-digit={d}.pkl' for d in data_cfg['digits']]}")

num_samples_per_digit = {d: (targets == d).sum().item() for d in data_cfg['digits']}
print(f"\n5. Expected Samples per Digit:")
for digit, count in num_samples_per_digit.items():
    print(f"   digit {digit}: {count} samples")

VERIFICATION: Key Metrics (for Reactivity Testing)

1. Data Shape & Properties:
   data.shape: torch.Size([5100, 1, 28, 28, 36])
   data.dtype: torch.float32
   targets.shape: torch.Size([5100])
   unique digits: [3]

2. Rotation Parameters:
   ANGLE_STEP: 10
   TIME_HORIZON: 36
   EPISODES_PER_IMAGE: 1
   Expected rotations per image: 3 (at 360°)

3. Data Statistics (before save):
   data min: 0.1000
   data max: 1.1000
   data mean: 0.2419
   data std: 0.3183

4. Output Configuration:
   SAVE_PATH: ../dataset/RotatingMNIST/
   digits to save: [3]
   files that will be created: ['RotatingMNIST-digit=3.pkl']

5. Expected Samples per Digit:
   digit 3: 5100 samples
